<a href="https://colab.research.google.com/github/christinafan-cell/MSSP607/blob/main/Assignments/Week_10/Week_10_AI_Review_of_Week9_Christina_Fan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Code Review and Revision: Image Downloader
**Student:** Christina Fan  
**Tool:** ChatGPT  
**Environment:** Google Colab

This notebook contains the original simplified image downloader, English AI review comments, recommendations, and revised Python code.

## 1. Request to the AI Tool
"Review my Python image downloader and recommend improvements to its clarity, reliability, and error handling. Annotate the original code in English, provide a revised version with English comments, and generate a downloadable Jupyter Notebook. Keep the code suitable for a beginner."

## 2. Original Code with AI Annotations
The original executable statements below are unchanged. Comments labeled **AI REVIEW** were added to explain specific improvements. This section is displayed as text so **Run all** executes only the revised program and does not download the images twice.

```python
# AI REVIEW 1: Keep all imports together; move tempfile to this section.
import requests
import shutil
from pathlib import Path
from google.colab import files

# Enter a photo category.
category = input("Enter a category (e.g., cats): ").strip()
if not category:
    raise ValueError("Please enter a category.")

# Search Flickr's public photo feed.
# AI REVIEW 2: Handle feed connection errors and invalid JSON explicitly.
response = requests.get(
    "https://www.flickr.com/services/feeds/photos_public.gne",
    params={"tags": category, "format": "json", "nojsoncallback": 1},
    timeout=30
)
response.raise_for_status()
# AI REVIEW 3: Validate the items field to give a clearer error for unexpected data.
photos = response.json()["items"]

# Create a new folder for this search.
import tempfile
folder = Path(tempfile.mkdtemp(prefix="photos_", dir="/content"))
count = 0

# Download every image returned by the feed.
# AI REVIEW 4: Skip duplicate or missing image URLs before downloading.
for i, photo in enumerate(photos, start=1):
    try:
        image = requests.get(photo["media"]["m"], timeout=30)
        image.raise_for_status()
# AI REVIEW 5: Check Content-Type and choose the matching file extension.
        # Saving every response as .jpg can mislabel PNG files or save an error page.
        (folder / f"photo_{i}.jpg").write_bytes(image.content)
        count += 1
# AI REVIEW 6: Also handle file-writing errors and print the error reason.
    except requests.RequestException:
        print(f"Could not download photo {i}.")

# AI REVIEW 7: Report failed and skipped items, and explain an empty result.
# Download the images as a ZIP file.
print(f"Downloaded {count} images.")
if count:
    zip_file = shutil.make_archive(str(folder), "zip", folder)
    files.download(zip_file)
```


## 3. Recommendations

| Review | Recommendation | Reason |
| --- | --- | --- |
| 1 | Group imports at the beginning. | Makes dependencies easier to identify. |
| 2 | Handle network errors and invalid feed data. | Gives a useful message if the search fails. |
| 3 | Validate the returned list. | Avoids an unexplained error when the response format changes. |
| 4 | Skip missing and duplicate URLs. | Avoids unnecessary downloads and missing-key errors. |
| 5 | Check the response type and preserve image formats. | Prevents saving non-image responses or labeling every file as JPEG. |
| 6 | Handle file-writing errors and show error details. | One failed download should not stop the remaining images. |
| 7 | Summarize downloaded, failed, and skipped items. | Makes the outcome easy to check. |
| 8 | Add a short delay between image requests. | Reduces the rate of requests sent to the server. |

**Retained strengths:** The original code already uses timeouts, checks HTTP errors with `raise_for_status()`, creates a separate folder for each run, and provides a convenient ZIP download.

**Scope limitation:** The program searches Flickr's public feed by tag and attempts to download every unique image returned. This limited feed is not a complete search of all matching photos on Flickr. The revision improves reliability but does not add full-site search or pagination. Neither Beautiful Soup nor browser automation is required for this JSON feed.

References: [Chapter 13](https://automatetheboringstuff.com/3e/chapter13.html) and [Flickr public feed documentation](https://www.flickr.com/services/feeds/docs/photos_public/).


## 4. Revised Code
Select **Runtime → Run all**, then enter an English tag such as `cats`, `flowers`, or `mountains`. No API key or input file is required. Colab includes `requests`; run `%pip install requests` if your environment does not.

The numbered comments below connect each change to the review above.


In [1]:
# REVIEW 1: Keep all imports together.
import requests
import shutil
import tempfile
import time
from pathlib import Path
from google.colab import files

category = input("Enter a category (e.g., cats): ").strip()
if not category:
    raise ValueError("Please enter a category.")

# REVIEWS 2-3: Check the search response and report clear errors.
try:
    response = requests.get(
        "https://www.flickr.com/services/feeds/photos_public.gne",
        params={"tags": category, "format": "json", "nojsoncallback": 1},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()
    if not isinstance(data, dict) or not isinstance(data.get("items"), list):
        raise ValueError("The feed did not return a list of photos.")
    photos = data["items"]
except (requests.RequestException, ValueError) as error:
    raise RuntimeError(f"Photo search failed: {error}") from error

folder = Path(tempfile.mkdtemp(prefix="photos_", dir="/content"))
downloaded = 0
failed = 0
skipped = 0
seen = set()

# REVIEW 5: Match the file extension to the response's image type.
extensions = {
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/gif": ".gif",
    "image/webp": ".webp"
}

for photo in photos:
    # REVIEW 4: Skip missing URLs and duplicates.
    media = photo.get("media", {}) if isinstance(photo, dict) else {}
    url = media.get("m") if isinstance(media, dict) else None
    if not isinstance(url, str) or not url or url in seen:
        skipped += 1
        continue
    seen.add(url)
    filename = None

    try:
        image = requests.get(url, timeout=30)
        image.raise_for_status()
        content_type = image.headers.get("Content-Type", "").split(";")[0].strip().lower()
        if content_type not in extensions or not image.content:
            raise ValueError("The response is empty or is not a supported image.")

        filename = folder / f"photo_{downloaded + 1:03d}{extensions[content_type]}"
        filename.write_bytes(image.content)
        downloaded += 1
        print(f"Saved {filename.name}")

    # REVIEW 6: Continue after network, response, or file-writing errors.
    except (requests.RequestException, OSError, ValueError) as error:
        failed += 1
        if filename is not None:
            filename.unlink(missing_ok=True)
        print(f"Download failed: {error}")

    # REVIEW 8: Space out requests.
    time.sleep(0.5)

# REVIEW 7: Show the outcome and download only when images were saved.
print(f"Returned: {len(photos)} | Downloaded: {downloaded} | Failed: {failed} | Skipped: {skipped}")
if downloaded:
    zip_file = shutil.make_archive(str(folder), "zip", folder)
    files.download(zip_file)
else:
    print("No images were saved. Try another category or check the error messages.")


Enter a category (e.g., cats): dogs
Saved photo_001.jpg
Saved photo_002.jpg
Saved photo_003.jpg
Saved photo_004.jpg
Saved photo_005.jpg
Saved photo_006.jpg
Saved photo_007.jpg
Saved photo_008.jpg
Saved photo_009.jpg
Saved photo_010.jpg
Saved photo_011.jpg
Saved photo_012.jpg
Saved photo_013.jpg
Saved photo_014.jpg
Saved photo_015.jpg
Saved photo_016.jpg
Saved photo_017.jpg
Saved photo_018.jpg
Saved photo_019.jpg
Saved photo_020.jpg
Returned: 20 | Downloaded: 20 | Failed: 0 | Skipped: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>